# Synthetic Gaussian Dataset
This notebook which walkthrough the implementation of a flow matching model from Gaussian noise to a Gaussian with a block covariance matrix.

## Setup

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
sys.path.insert(0, os.path.abspath('.'))

from gaussian_data  import get_gaussian_dataset, DIM, Sigma, evaluate
from gaussian_model import MLP_Residual, cfm_loss, sample, sincos_embed, SIGMA_MIN

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"DIM:    {DIM}")

## Data

The data is preset to be a flattened vector of a 32 by 32 grid (1024 dim), hence, we go from $x\sim \mathcal N(0,1)$ to $x\sim \mathcal N(0,\Sigma)$ where $x\in \mathbb R^{1024}$ and $\Sigma\in \mathbb R^{1024\times 1024}$ is a covariance block matrix. We have chosen to do 4 blocks (so if we were to visualise the 32 by 32 grid, it resemble a 4 by 4 checkerboard). Hence, the covariance matrix resembles
$$
\Sigma = (1-\rho)I_{1024\times 1024}+\rho
\begin{pmatrix}
\mathbf 1_{256 \times 256} & \mathbf 0_{256\times 256} & \mathbf 0_{256\times 256} & \mathbf 0_{256\times 256} \\
\mathbf 0_{256\times 256} &\mathbf 1_{256 \times 256} & \mathbf 0_{256\times 256} & \mathbf 0_{256\times 256} \\
\mathbf 0_{256\times 256} & \mathbf 0_{256\times 256} & \mathbf 1_{256 \times 256} & \mathbf 0_{256\times 256} \\
\mathbf 0_{256\times 256} & \mathbf 0_{256\times 256} & \mathbf 0_{256\times 256} & \mathbf 1_{256 \times 256} \\
\end{pmatrix}
$$
where $\rho<1$ is the intensity, (higher $\rho$ leads to more intense checkerboard pattern). We arbitrarily use $\rho=0.85$ for our generated data.

In [ ]:
data = torch.from_numpy(get_gaussian_dataset(n_samples=10000)).to(device)
print(data.shape)

It may be helpful to visualise what $x\sim \mathcal N(0,\Sigma)$ looks like.

In [ ]:
def show_sample(x: torch.Tensor):
    img = x.cpu().numpy().reshape(32, 32)
    plt.imshow(img, cmap='gray', interpolation='nearest')
    plt.axis('off')
    plt.show()

show_sample(data[2])

## Forward Process

We will use conditional flow matching as it pertains to  the paper by Lipman et al. Consider a probability density path $\rho:[0,1]\times \mathbb R^d\to \mathbb R_{>0}$ which satisfy $\int p_t(x)dx=1$. Our goal is to learn a vector field $v:[0,1]\times \mathbb R^d\to \mathbb R^d$ to construct a flow $\phi:[0,1]\times \mathbb R^d\to \mathbb R^d$ defined by
$$
\frac{d}{dt}\phi_t(x)=v_t(\phi_t(x))
$$
where
$$
p_t=[\phi_t]_\star p_0
$$

Given that we have defined a probability path $p_t(x)$ and a corresponding vector field $u_t(x)$ which generates $p_t(x)$, a desireable objective function would be
$$
\mathcal L_{FM}(\theta)=\mathbb E_{t\sim \mathcal U(0,1), x\sim p_t(x)}||v_t(x;\theta)-u_t(x)||^2
$$
where we must choose a probability path which satisfy $p_1(x)\approx q(x)$ which is the target distribution, and $p_0(x)\sim p_0$ which is the base distribution.

**Reference:** Lipman, Y., Chen, R. T. Q., Ben-Hamu, H., Nickel, M., & Le, M. (2022). *Flow Matching for Generative Modeling*. Preprint. Meta AI (FAIR) & Weizmann Institute of Science.

### Conditional Flow Matching

Due to intractability, we consider a different objective, which would yield the same optimal vector field by considering the marginals of the generating field instead.

We consider conditional probability paths of the form

$$
p_t(x \mid x_1) = \mathcal{N}(x \mid \mu_t(x_1), \sigma_t(x_1)^2 I)
$$

where we set $\mu_0(x_1) = 0$, $\sigma_0(x_1) = 1$ so that $p_0(x) = \mathcal{N}(x \mid 0, I)$, and $\mu_1(x_1) = x_1$, $\sigma_1(x_1) = \sigma_{\min}$ so that $p_1(x \mid x_1)$ is concentrated at $x_1$. This gives the **Conditional Flow Matching (CFM)** objective:

$$
\mathcal{L}_{\text{CFM}}(\theta) = \mathbb{E}_{t \sim \mathcal{U}(0,1),\, x_1 \sim q(x_1),\, x \sim p_t(x \mid x_1)} \left\| v_t(x;\theta) - u_t(x \mid x_1) \right\|^2
$$

which has identical gradients to $\mathcal{L}_{\text{FM}}$.

### Optimal Transport Path

We use the **OT conditional path**, where the mean and std change linearly in time:

$$
\mu_t(x_1) = t x_1, \qquad \sigma_t(x_1) = 1 - (1 - \sigma_{\min})t
$$

This gives the conditional flow $\psi_t(x_0) = (1 - (1-\sigma_{\min})t)\, x_0 + t x_1$ and the corresponding vector field:

$$
u_t(x \mid x_1) = \frac{x_1 - (1 - \sigma_{\min})x_0}{1 - (1-\sigma_{\min})t}
$$

The CFM loss under this path becomes:

$$
\mathcal{L}_{\text{CFM}}(\theta) = \mathbb{E}_{t,\, x_1 \sim q(x_1),\, x_0 \sim p(x_0)} \left\| v_t(\psi_t(x_0);\theta) - \left(x_1 - (1-\sigma_{\min})x_0\right) \right\|^2
$$

with $\sigma_{\min} = 10^{-4}$ throughout.

## Model

We use a residual MLP $v_\theta : \mathbb{R}^{1024} \times [0,1] \to \mathbb{R}^{1024}$ to learn the vector field. At each hidden layer, the timestep $t$ is injected via a learned projection of the sinusoidal time embedding, allowing each layer to respond to the noise level independently:

$$h \leftarrow h + \sigma\bigl(W_\ell h + P_\ell e_t\bigr)$$

In [ ]:
model = MLP_Residual(
    input_size     = DIM,   # 1024
    hidden_size    = 512,
    amount_layers  = 4,
    output_size    = DIM,   # 1024
    time_dimension = 64,
).to(device)

## Training

We minimise the OT-CFM loss using Adam. At each gradient step we sample a random minibatch from the dataset, draw fresh base samples $x_0 \sim \mathcal{N}(0, I)$ and timesteps $t \sim \mathcal{U}(0,1)$ inside `cfm_loss`, and update the model weights.

In [ ]:
from tqdm import tqdm

N_STEPS    = 10000
BATCH_SIZE = 256
LR         = 3e-4

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
losses    = []

pbar = tqdm(range(N_STEPS))
for step in pbar:
    idx  = torch.randperm(len(data))[:BATCH_SIZE]
    x1   = data[idx]
    loss = cfm_loss(model, x1)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    pbar.set_postfix(loss=f"{loss.item():.5f}")

plt.plot(losses)
plt.xlabel("gradient step")
plt.ylabel("CFM loss")
plt.title("Training Curve")
plt.grid(alpha=0.3)
plt.show()

## Sampling

Once trained, we generate samples by integrating the learned vector field $v_\theta(x_t, t)$ forward from $t=0$ to $t=1$ using Euler's method:
$$x_{t+\Delta t} = x_t + v_\theta(x_t, t) \cdot \Delta t, \qquad \Delta t = \frac{1}{T}$$
starting from $x_0 \sim \mathcal{N}(0, I)$. After $T$ steps we arrive at $x_1 \approx \mathcal{N}(0, \Sigma)$.

In [ ]:
generated = sample(model, n_samples=1000, n_steps=100, device=device)

## Evaluation

Since the true distribution $\mathcal{N}(0, \Sigma)$ is known exactly, we can directly verify whether the model learned it correctly using two metrics.

**Covariance Error**: we compare the empirical covariance of generated samples
to the true $\Sigma$:

$$\text{Frob} = \|\Sigma_{\text{true}} - \widehat{\Sigma}\|_F, \qquad \widehat{\Sigma} = \frac{1}{N}\sum_{i=1}^N \hat{x}_i \hat{x}_i^\top$$

If the model learned the correct distribution, $\widehat{\Sigma} \approx \Sigma$ and the Frobenius error should be close to $0$.

**Wasserstein Distance**: we compute the 1D Wasserstein distance between generated and true samples independently for each of the 1024 dimensions and average across all dimensions:

$$\overline{W}_1 = \frac{1}{d}\sum_{j=1}^{d} W_1\bigl(\hat{x}^{(j)},\, x^{(j)}\bigr)$$

where $x^{(j)}$ denotes the $j$-th marginal. A well-trained model should drive both metrics close to $0$.

In [ ]:
metrics = evaluate(generated)

print("Covariance Error")
print(f"  mean absolute error : {metrics['mean_abs_error']:.6f}")
print(f"  max absolute error  : {metrics['max_abs_error']:.6f}")
print(f"  Frobenius error     : {metrics['frobenius_error']:.4f}")

print("Wasserstein Distance")
print(f"  mean (across dims)  : {metrics['mean_wasserstein']:.6f}")
print(f"  max  (across dims)  : {metrics['max_wasserstein']:.6f}")

## Visualizing the Flow

We visualize the learned flow by tracking how a set of base samples $x_0 \sim \mathcal{N}(0, I)$ evolve through the vector field at intermediate timesteps $t \in \{0, 0.25, 0.5, 0.75, 1.0\}$. At $t=0$ the samples look like pure noise and at $t=1$ they should exhibit the block covariance structure of $\mathcal{N}(0, \Sigma)$.

In [ ]:
model.eval()
timesteps  = [0.0, 0.25, 0.5, 0.75, 1.0]
n_show     = 2
n_steps    = 100
dt         = 1.0 / n_steps

# run euler integration and snapshot at each timestep
x = torch.randn(n_show, DIM, device=device)
snapshots = {0.0: x.clone()}

for i in range(n_steps):
    t_val = i / n_steps
    t     = torch.full((n_show,), t_val, device=device)
    with torch.no_grad():
        x = x + model(x, t) * dt
    for snap_t in timesteps[1:]:
        if abs(t_val + dt - snap_t) < dt / 2:
            snapshots[snap_t] = x.clone()

# plot
fig, axes = plt.subplots(n_show, len(timesteps), figsize=(14, 3 * n_show))
n_blocks  = 32 // 8

for row in range(n_show):
    for col, t_val in enumerate(timesteps):
        img = snapshots[t_val][row].cpu().numpy().reshape(32, 32)
        axes[row, col].imshow(img, cmap="gray", interpolation="nearest")
        for b in range(1, n_blocks):
            axes[row, col].axhline(b * 8 - 0.5, color="cyan", linewidth=0.8, alpha=0.6)
            axes[row, col].axvline(b * 8 - 0.5, color="cyan", linewidth=0.8, alpha=0.6)
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(f"t = {t_val}", color="black", fontsize=11)

plt.suptitle("Flow trajectory", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()